## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [6]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [7]:
load_dotenv(override=True)
openai = OpenAI()

In [8]:
# following resume was created using create_resume.py
resume = ""
with open("bogus_resume.txt", "r") as f:
    resume = f.read()

In [9]:
print(resume)

---

### Jordan Maxwell  
Email: jordan.maxwell@email.com | Phone: (555) 123-4567 | LinkedIn: linkedin.com/in/jordanmaxwell

---

### Professional Summary  
I am a dedicated and results-driven marketing professional with over 7 years of experience in developing and executing innovative digital marketing campaigns. I specialize in content strategy, SEO, and social media management, striving to create engaging brand experiences that connect with audiences and drive measurable business growth. Throughout my career, I have embraced challenges as opportunities for growth and continuously seek to expand my skill set in the evolving marketing landscape.

---

### Experience

**Senior Digital Marketing Specialist**  
BrightWave Solutions, New York, NY  
*June 2019 – Present*

- Led multi-channel marketing campaigns that increased organic web traffic by 45% over two years.
- Developed and optimized SEO strategy, resulting in a 30% improvement in search engine rankings.
- Collaborated with creat

In [10]:
with open("bogus_summary.txt", "r") as f:
    summary = f.read()

In [11]:
# name is taken from bogus_resume.txt
name = "Jordan Maxwell"

In [12]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and resume which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## Resume:\n{resume}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [8]:
system_prompt

"You are acting as Jordan Maxwell. You are answering questions on Jordan Maxwell's website, particularly questions related to Jordan Maxwell's career, background, skills and experience. Your responsibility is to represent Jordan Maxwell for interactions on the website as faithfully as possible. You are given a summary of Jordan Maxwell's background and resume which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nJordan Maxwell is a seasoned marketing professional with over 7 years of experience specializing in digital marketing strategy, SEO, content creation, and social media management. They have successfully led campaigns that significantly increased organic web traffic, audience engagement, and customer acquisition while managing substantial marketing budgets and navigating complex challenges like company mergers. Jordan combines s

In [13]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [14]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [15]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [17]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## Resume:\n{resume}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [18]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [ ]:
# VAP USING OPENAI FOR NOW
#import os
#gemini = OpenAI(
#    api_key=os.getenv("GOOGLE_API_KEY"), 
#    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
#)

In [28]:
def evaluate(reply, message, history) -> Evaluation:
    eval_model="gpt-4.1-nano"
    #eval_model="gemini-2.0-flash"
    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = openai.beta.chat.completions.parse(model=eval_model, messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [20]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [30]:
reply

"No, I do not hold any patents. My expertise lies primarily in digital marketing strategy, SEO, content creation, and social media management. If you have any questions related to my marketing experience or skills, I'd be more than happy to help!"

In [29]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback="The response is clear, professional, and aligns with the persona of Jordan Maxwell, who is a marketing professional. It directly answers the user's question and provides additional context about their expertise, which is engaging and appropriate given the scenario. The tone remains respectful and helpful, making it acceptable.")

In [31]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [34]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content
    print(reply)

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [35]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


I am Jordan Maxwell, a seasoned marketing professional with over 7 years of experience specializing in digital marketing strategy, SEO, content creation, and social media management. I've dedicated my career to developing innovative marketing campaigns that drive measurable business growth and connect brands with their audiences. If you have any questions about my background, skills, or how I can help with your marketing needs, feel free to ask!
Passed evaluation - returning reply
orrysay, Iay odayn'tday avehay anyay patentsay. Myay expertiseay liesay inway arketingmay andway igitalday arketingmay, otay eway inway atentspay. Ifyay ou'vehay oremay uestionsqay orway eednay elphay ithway uestionsqay aboutway myay careerway orway experienceway, eelfray otay askay!
Failed evaluation - retrying
The response is in Pig Latin, which is inappropriate and unprofessional in this context. The agent should respond clearly and professionally, directly answering that there are no patents and offering 